# 01 · Kerala school list — KITE Sametham

**Source:** KITE Sametham – Kerala School Data Bank (Kerala Infrastructure and Technology for Education,
General Education Department, Government of Kerala), https://sametham.kite.kerala.gov.in/

**Pages:** the 14 public district lists `search/districtWiseSchools/1` … `/14`. Each row: school code
(section prefix LP/UP/HS), name, block, local body, parliament and assembly constituency, finance type
(Government / Aided / Unaided Recognised), class range, phone, and the Google Maps link Sametham
publishes for the school (lat/lon).

**Method:** automated, rate-limited download (1.2 s between pages, no login, the site has no robots.txt).
Raw pages are kept unchanged in `data/raw/sametham_district_pages/`.

**Official input of the project:** the list downloaded on **2026-10-26**
(`data/raw/sametham/sametham_kerala_schools_all_2026-10-26.csv`, 12,760 schools; 11,768 Government +
Aided). Sametham is a live directory, so this notebook writes a rebuilt copy
(`data/interim/schools/sametham_school_list_rebuilt.csv`) and reports what changed since that date.

## Imports and settings

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))
import html
import re
import time

import pandas as pd
import requests

RAW_DIR = PROJECT / "data" / "raw" / "sametham_district_pages"
OFFICIAL = PROJECT / "data" / "raw" / "sametham" / "sametham_kerala_schools_all_2026-10-26.csv"
OUT_CSV = PROJECT / "data" / "interim" / "schools" / "sametham_school_list_rebuilt.csv"
URL = "https://sametham.kite.kerala.gov.in/search/districtWiseSchools/{}"   # districts 1..14
COLS = ["district_page", "district", "school_code", "school_name", "block", "local_body", "parliament",
        "assembly", "finance_type", "level", "phone", "lat", "lon"]

In [ ]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio"], check=True)

### `fetch()`

Download a district page once and keep it unchanged.

In [2]:
def fetch(n, session):
    """Download district page n once (kept unchanged in RAW_DIR), wait 1.2 s between requests."""
    f = RAW_DIR / f"district_{n}.html"
    if not f.exists():
        r = session.get(URL.format(n), timeout=120)
        r.raise_for_status()
        f.write_text(r.text, encoding="utf-8")
        time.sleep(1.2)
    return f.read_text(encoding="utf-8")

### `cells()`

Read the text of each table cell.

In [3]:
def cells(h):
    """Text of every <td>/<th> cell in an HTML fragment."""
    out = []
    for c in re.findall(r"(?s)<t[dh][^>]*>(.*?)</t[dh]>", re.sub(r"(?s)<!--.*?-->", "", h)):
        c = re.sub(r"<br\s*/?>", " ", c)
        out.append(re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", c))).strip())
    return out

### `parse()`

Turn the district table into rows.

In [4]:
def parse(raw, n):
    """One row per school: code, name, block, local body, constituencies, finance type, classes, phone, lat/lon."""
    district = re.search(r"Revenue District\s*:\s*([^<]+)<", raw).group(1).strip()
    rows = []
    for tr in re.findall(r"(?s)<tr>(.*?)</tr>", raw.split("<tbody>", 1)[1]):
        c = cells(tr)
        if len(c) < 12 or not c[0].isdigit():
            continue
        loc = re.search(r"maps\?q=(-?[\d.]+),(-?[\d.]+)", tr)   # Sametham's Google Maps link holds the lat/lon
        rows.append([n, district, re.sub(r"^([A-Z]+):\s*", r"\1: ", c[1]), c[2], c[3] or None,
                     re.sub(r"\s*\(", " (", c[4]), c[5], c[6], c[7], c[8], c[9] or None,
                     loc.group(1) if loc else None, loc.group(2) if loc else None])
    return rows

### `main()`

In [5]:
def main():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    session.headers["User-Agent"] = "msc-spatial-inequality-study/0.1 (academic project)"
    df = pd.DataFrame([r for n in range(1, 15) for r in parse(fetch(n, session), n)], columns=COLS)
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8")
    print(f"schools: {len(df)} | Government + Aided: {df['finance_type'].isin(['Government', 'Aided']).sum()}")
    print("by finance type:", df["finance_type"].value_counts().to_dict())
    return df

## Run

In [6]:
df = main()

schools: 12760 | Government + Aided: 11768
by finance type: {'Aided': 7074, 'Government': 4694, 'Unaided Recognised': 992}


## Check against the official 2026-10-26 list

In [7]:
old = pd.read_csv(OFFICIAL, dtype=str)
new = df.astype(str).replace({"None": None, "nan": None})
print(f"official 2026-10-26: {len(old)} schools | rebuilt now: {len(new)}")
print("only in official:", sorted(set(old.school_code) - set(new.school_code))[:20])
print("only in rebuilt: ", sorted(set(new.school_code) - set(old.school_code))[:20])
m = old.merge(new, on="school_code", suffixes=("_old", "_new"))
for col in ["school_name", "block", "local_body", "finance_type", "level", "phone", "lat", "lon"]:
    a, b = m[f"{col}_old"].fillna("").str.strip(), m[f"{col}_new"].fillna("").str.strip()
    print(f"changed {col:12s}: {(a != b).sum()}")

official 2026-10-26: 12760 schools | rebuilt now: 12760
only in official: []
only in rebuilt:  []
changed school_name : 2
changed block       : 1
changed local_body  : 1
changed finance_type: 0
changed level       : 0
changed phone       : 53
changed lat         : 0
changed lon         : 0
